# Module 2: Data Processing with Python

**BigData Analytics Course — AKTN Magister Program**

---

## Learning Objectives
1. Manipulate large arrays and matrices with **NumPy**.
2. Load, clean, and transform tabular data with **Pandas**.
3. Build a basic **ETL (Extract–Transform–Load)** pipeline in Python.
4. Handle missing values, duplicates, and outliers.
5. Apply grouping, aggregation, merging, and reshaping operations.

**Estimated time:** 120 minutes  
**Prerequisites:** Module 1, basic Python (functions, lists, dicts)

In [ ]:
# Install any missing packages (pre-installed on Colab)
# !pip install pandas numpy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"NumPy  : {np.__version__}")
print(f"Pandas : {pd.__version__}")
print("✅ Ready!")

## 1. NumPy Fundamentals

NumPy's `ndarray` is the foundation of nearly all scientific Python work. It stores homogeneous, fixed-type data in a contiguous memory block, enabling vectorised operations that are **10–100× faster** than equivalent Python loops.

In [ ]:
# ── 1.1 Array creation ──────────────────────────────────────────────────────
a1d = np.array([10, 20, 30, 40, 50])            # 1-D array
a2d = np.arange(1, 13).reshape(3, 4)            # 3×4 matrix
zeros = np.zeros((2, 3))
ones  = np.ones((2, 3))
ident = np.eye(4)                               # identity matrix
rand  = np.random.default_rng(42).random((3, 3))

print("1-D array:\n", a1d)
print("\n2-D array (3×4):\n", a2d)
print("\nIdentity 4×4:\n", ident)
print("\nRandom 3×3:\n", rand.round(3))

In [ ]:
# ── 1.2 Vectorised operations vs Python loop ────────────────────────────────
import time

N = 10_000_000
data = np.random.rand(N)

t0 = time.perf_counter()
result_loop = [x ** 2 for x in data]   # Python loop
t_loop = time.perf_counter() - t0

t0 = time.perf_counter()
result_numpy = data ** 2                # NumPy vectorised
t_numpy = time.perf_counter() - t0

print(f"Python loop  : {t_loop:.3f}s")
print(f"NumPy vector : {t_numpy:.4f}s")
print(f"Speed-up     : {t_loop / t_numpy:.1f}×")

In [ ]:
# ── 1.3 Linear algebra — essential for ML ──────────────────────────────────
A = np.array([[2, 1], [5, 3]])
b = np.array([4, 7])

# Solve Ax = b
x = np.linalg.solve(A, b)
print("Solution x:", x)
print("Verification Ax ≈ b:", np.allclose(A @ x, b))

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eig(A)
print("\nEigenvalues :", eigenvalues.round(4))
print("Eigenvectors:\n", eigenvectors.round(4))

## 2. Pandas for Data Manipulation

Pandas provides two key data structures:
- **Series** — 1-D labelled array
- **DataFrame** — 2-D labelled table (think: supercharged spreadsheet)

We will work with a synthetic **e-commerce transactions** dataset throughout this module.

In [ ]:
# ── 2.1 Generate a synthetic e-commerce dataset ─────────────────────────────
rng = np.random.default_rng(42)
n = 1_000

categories = ['Electronics', 'Clothing', 'Books', 'Home & Garden', 'Sports']
regions    = ['North', 'South', 'East', 'West']

df = pd.DataFrame({
    'order_id'    : range(1001, 1001 + n),
    'date'        : pd.date_range('2024-01-01', periods=n, freq='h'),
    'category'    : rng.choice(categories, n),
    'region'      : rng.choice(regions, n),
    'quantity'    : rng.integers(1, 20, n),
    'unit_price'  : rng.uniform(5, 500, n).round(2),
    'discount_pct': rng.choice([0, 5, 10, 15, 20], n),
    'customer_age': rng.integers(18, 70, n),
})

# Introduce intentional data quality issues
df.loc[rng.choice(n, 30, replace=False), 'unit_price'] = np.nan   # missing prices
df.loc[rng.choice(n, 10, replace=False), 'quantity']  = -1        # invalid quantity
df = pd.concat([df, df.iloc[:5]], ignore_index=True)               # duplicate rows

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# ── 2.2 Exploratory overview ────────────────────────────────────────────────
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Descriptive Statistics ===")
df.describe()

## 3. Data Cleaning Pipeline

In [ ]:
def clean_ecommerce(raw: pd.DataFrame) -> pd.DataFrame:
    """Clean the e-commerce DataFrame — a reusable pipeline function."""
    df = raw.copy()

    # Step 1: Remove exact duplicates
    n_before = len(df)
    df.drop_duplicates(inplace=True)
    print(f"[1] Removed {n_before - len(df)} duplicate rows")

    # Step 2: Fix invalid quantity values
    invalid_qty = (df['quantity'] <= 0).sum()
    df = df[df['quantity'] > 0]
    print(f"[2] Removed {invalid_qty} rows with invalid quantity")

    # Step 3: Impute missing unit_price with category median
    missing_price = df['unit_price'].isnull().sum()
    df['unit_price'] = df.groupby('category')['unit_price'].transform(
        lambda s: s.fillna(s.median())
    )
    print(f"[3] Imputed {missing_price} missing unit_price values")

    # Step 4: Derive new columns
    df['revenue'] = (
        df['quantity'] * df['unit_price'] * (1 - df['discount_pct'] / 100)
    ).round(2)
    df['month'] = df['date'].dt.to_period('M').astype(str)
    print(f"[4] Added 'revenue' and 'month' columns")

    # Step 5: Reset index
    df.reset_index(drop=True, inplace=True)
    print(f"[5] Final shape: {df.shape}")
    return df

df_clean = clean_ecommerce(df)
df_clean.head()

## 4. Aggregation & Grouping

In [ ]:
# ── 4.1 Revenue by category ─────────────────────────────────────────────────
rev_by_cat = (
    df_clean
    .groupby('category', as_index=False)
    .agg(
        total_revenue=('revenue', 'sum'),
        avg_order_value=('revenue', 'mean'),
        num_orders=('order_id', 'count')
    )
    .sort_values('total_revenue', ascending=False)
)
print(rev_by_cat.to_string(index=False))

In [ ]:
# ── 4.2 Monthly revenue trend ───────────────────────────────────────────────
monthly = (
    df_clean
    .groupby('month')['revenue']
    .sum()
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart — revenue by category
axes[0].bar(rev_by_cat['category'], rev_by_cat['total_revenue'],
            color='#2196F3', edgecolor='white')
axes[0].set_title('Total Revenue by Category', fontsize=13)
axes[0].set_ylabel('Revenue ($)')
axes[0].tick_params(axis='x', rotation=20)

# Line chart — monthly trend
axes[1].plot(monthly['month'], monthly['revenue'],
             marker='o', color='#4CAF50', linewidth=2)
axes[1].set_title('Monthly Revenue Trend', fontsize=13)
axes[1].set_ylabel('Revenue ($)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Merging, Pivoting & Reshaping

In [ ]:
# ── 5.1 Pivot table — region × category revenue ─────────────────────────────
pivot = df_clean.pivot_table(
    values='revenue',
    index='region',
    columns='category',
    aggfunc='sum',
    fill_value=0
)
print("Revenue Pivot Table (region × category):")
print(pivot.round(0).astype(int))

In [ ]:
# ── 5.2 Merge: enrich orders with a 'category info' lookup table ────────────
category_info = pd.DataFrame({
    'category'    : categories,
    'tax_rate_pct': [11, 10, 5, 10, 12],
    'department'  : ['Tech', 'Fashion', 'Media', 'Home', 'Outdoor'],
})

df_enriched = df_clean.merge(category_info, on='category', how='left')
df_enriched['revenue_after_tax'] = (
    df_enriched['revenue'] * (1 + df_enriched['tax_rate_pct'] / 100)
).round(2)

print(f"Enriched shape: {df_enriched.shape}")
df_enriched[['order_id', 'category', 'revenue', 'tax_rate_pct', 'revenue_after_tax']].head(8)

## 6. ETL Pipeline — Putting It All Together

In [ ]:
import io

def etl_pipeline(raw_data: pd.DataFrame, lookup: pd.DataFrame) -> pd.DataFrame:
    """
    Full ETL pipeline:
      Extract  → receive raw DataFrames
      Transform → clean, enrich, aggregate
      Load     → return analysis-ready DataFrame
    """
    print("[EXTRACT]  Raw rows:", len(raw_data))

    # Transform
    cleaned   = clean_ecommerce(raw_data)
    enriched  = cleaned.merge(lookup, on='category', how='left')
    enriched['revenue_after_tax'] = (
        enriched['revenue'] * (1 + enriched['tax_rate_pct'] / 100)
    ).round(2)

    # Load (summary table)
    summary = (
        enriched.groupby(['department', 'region'], as_index=False)
        .agg(
            orders        = ('order_id', 'count'),
            gross_revenue = ('revenue', 'sum'),
            net_revenue   = ('revenue_after_tax', 'sum'),
        )
        .round(2)
    )

    print("[LOAD]     Output rows:", len(summary))
    return summary

report = etl_pipeline(df, category_info)
print("\n=== Final Report ===")
print(report.to_string(index=False))

## 7. Summary

| Topic | Key Points |
|-------|------------|
| NumPy | Vectorised operations — avoid Python loops; essential for numerical computing |
| Pandas | DataFrame: load, inspect, clean, reshape, aggregate, merge |
| Data Quality | Handle NaN, duplicates, invalid values before analysis |
| ETL | Extract → Transform → Load — the backbone of data engineering |

## 📝 Exercises

1. **NumPy:** Create a 5×5 magic square and verify that all row, column, and diagonal sums are equal.
2. **Pandas:** Load the [Titanic dataset](https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv) and calculate the survival rate by passenger class and sex.
3. **ETL:** Extend the pipeline to export the final report as a CSV file to `/content/drive/MyDrive/` (if using Colab with Drive mounted).

---
⬅️ [Module 1 — Introduction to Big Data](Module_01_Introduction_to_Big_Data.ipynb)  
➡️ [Module 3 — Data Visualization](Module_03_Data_Visualization.ipynb)